# GNO2d - the Graph Neural Operator

*Walkthrough notebook for `neural_operators/models/gno.py`.*

FNO needs a regular grid (FFT) and a translation-invariant kernel. The GNO (paper Section 4.1) handles a
general kernel $\kappa(x,y)$ on arbitrary points by treating the grid as a **graph** and doing message passing.
Two ideas keep the $O(J^2)$ integral affordable:

1. **Truncation** - only integrate over points within radius $r$ of $x$ (a sparse, local sum).
2. **Nystrom subsampling** - only use a fixed random subset of $J'\ll J$ **key points** as message sources.

A GNO layer then computes, for every query point $x_j$,

$$u(x_j)=\frac1{|N(x_j)|}\sum_{y\in N(x_j)}\kappa(x_j,y)\,v(y),$$

where $\kappa$ is the learned `KernelMLP` producing one matrix per edge, and we take the **mean** over neighbours.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from neural_operators.layers.kernel_nn import KernelMLP

## Coordinates

Flattens a `size_x x size_y` grid on $[0,1]^2$ into an `(N, 2)` list of points.

In [ ]:
def make_grid_coords(size_x, size_y, device=None):
    gx = torch.linspace(0, 1, size_x, device=device)
    gy = torch.linspace(0, 1, size_y, device=device)
    xx, yy = torch.meshgrid(gx, gy, indexing="ij")
    return torch.stack((xx.reshape(-1), yy.reshape(-1)), dim=-1)  # (N, 2)

## Building the graph

`edge_index[0]` = source (key) node, `edge_index[1]` = target (query) node, for every pair closer than
`radius`. `cdist` computes all pairwise distances, so this is $O(n_{query}\cdot n_{key})$ memory - fine at
the sizes used here, and it is only built once and cached.

In [ ]:
def build_radius_graph(query_coords, key_coords, radius):
    """Returns edge_index (2, E) with row 0 = key (source) index into key_coords,
    row 1 = query (target) index into query_coords, for all pairs within `radius`."""
    dist = torch.cdist(query_coords, key_coords)  # (n_query, n_key)
    mask = dist <= radius
    target, source = mask.nonzero(as_tuple=True)
    edge_index = torch.stack((source, target), dim=0)
    return edge_index

## One message-passing layer

Read `forward` top to bottom: kernel network -> a matrix per edge; gather source features; multiply
(`einsum`) to get messages; `index_add_` sums messages into their target node; divide by the neighbour
count for the mean. Mean aggregation keeps magnitudes stable regardless of how many neighbours a node has.

In [ ]:
class GNOLayer(nn.Module):
    def __init__(self, width, coord_dim=2, kernel_hidden=64):
        super().__init__()
        self.kernel_net = KernelMLP(2 * coord_dim, dv_in=width, dv_out=width, hidden=kernel_hidden)
        self.local_linear = nn.Linear(width, width)  # W_t, the local (pointwise) term in Eq. (10)
        self.bias = nn.Parameter(torch.zeros(width))

    def forward(self, v, edge_index, edge_features, num_query):
        """v: (batch, n_key, width) node features at the key points.
        edge_index: (2, E) = (source in key set, target in query set).
        edge_features: (E, 2*coord_dim) = concat(x_target, y_source) coordinates.
        num_query: number of query points (target index range).
        """
        source, target = edge_index[0], edge_index[1]
        batch = v.shape[0]

        kappa = self.kernel_net(edge_features)          # (E, width, width)
        v_src = v[:, source, :]                          # (batch, E, width)
        messages = torch.einsum("eoi,bei->beo", kappa, v_src)  # (batch, E, width)

        agg = torch.zeros(batch, num_query, v.shape[-1], device=v.device, dtype=v.dtype)
        counts = torch.zeros(num_query, device=v.device, dtype=v.dtype)
        agg.index_add_(1, target, messages)
        counts.index_add_(0, target, torch.ones_like(target, dtype=v.dtype))
        counts = counts.clamp(min=1.0).view(1, -1, 1)
        agg = agg / counts  # mean aggregation, matches Eq. after (16)

        # local linear term W_t applied to v at the *query* points -- if key set ==
        # query set (as in our stacked-layer setup) this is just self.local_linear(v).
        return agg, kappa  # kappa returned only for introspection/teaching purposes

## The whole network

`_get_graph` builds the random key subset and the radius graph **once per resolution** and caches it.
`forward` lifts per-point inputs, runs `n_layers` of (graph message + local linear term), then projects to the
output - same lift/project structure as the FNO.

In [ ]:
class GNO2d(nn.Module):
    def __init__(
        self,
        in_channels=3,
        out_channels=1,
        width=24,
        n_layers=4,
        radius=0.08,
        n_subsample=512,
        seed=0,
    ):
        super().__init__()
        self.width = width
        self.radius = radius
        self.n_subsample = n_subsample
        self.seed = seed

        self.lift = nn.Linear(in_channels, width)
        self.gno_layers = nn.ModuleList([GNOLayer(width) for _ in range(n_layers)])
        self.local_linears = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.project1 = nn.Linear(width, 128)
        self.project2 = nn.Linear(128, out_channels)

        self._graph_cache = {}

    def _get_graph(self, size_x, size_y, device):
        key = (size_x, size_y, str(device))
        if key not in self._graph_cache:
            coords = make_grid_coords(size_x, size_y, device=device)
            n_points = coords.shape[0]
            g = torch.Generator(device="cpu").manual_seed(self.seed)
            n_sub = min(self.n_subsample, n_points)
            key_idx = torch.randperm(n_points, generator=g)[:n_sub].to(device)
            key_coords = coords[key_idx]

            edge_index = build_radius_graph(coords, key_coords, self.radius)
            edge_features = torch.cat(
                (coords[edge_index[1]], key_coords[edge_index[0]]), dim=-1
            )  # concat(x_target, y_source)
            self._graph_cache[key] = (coords, key_idx, edge_index, edge_features)
        return self._graph_cache[key]

    def forward(self, a):
        if a.dim() == 3:
            a = a.unsqueeze(-1)
        batch, size_x, size_y = a.shape[0], a.shape[1], a.shape[2]
        device = a.device

        coords, key_idx, edge_index, edge_features = self._get_graph(size_x, size_y, device)
        n_points = size_x * size_y

        a_flat = a.reshape(batch, n_points, -1)
        coords_b = coords.unsqueeze(0).expand(batch, -1, -1)
        x = torch.cat((a_flat, coords_b), dim=-1)
        v = self.lift(x)  # (batch, n_points, width)

        for t, (gno_layer, local_linear) in enumerate(zip(self.gno_layers, self.local_linears)):
            v_key = v[:, key_idx, :]
            agg, _ = gno_layer(v_key, edge_index, edge_features, num_query=n_points)
            v = agg + local_linear(v)
            if t < len(self.gno_layers) - 1:
                v = F.gelu(v)

        out = F.gelu(self.project1(v))
        out = self.project2(out)
        return out.view(batch, size_x, size_y)

## Try it

Look at the graph on its own first, then run the model.

In [ ]:
import matplotlib.pyplot as plt

coords = make_grid_coords(16, 16)
edge_index = build_radius_graph(coords, coords, radius=0.15)
print("points:", coords.shape[0], "| edges:", edge_index.shape[1],
      "| average neighbours:", edge_index.shape[1] / coords.shape[0])

# Show the neighbourhood of one query point
q = 120
nbrs = edge_index[0][edge_index[1] == q]
plt.figure(figsize=(4, 4))
plt.scatter(*coords.T, s=8, c="lightgray")
plt.scatter(*coords[nbrs].T, s=20, c="tab:blue", label="neighbours")
plt.scatter(*coords[q:q+1].T, s=60, c="tab:red", label="query point")
plt.legend(); plt.title("Radius graph neighbourhood"); plt.show()

In [ ]:
from neural_operators.utils import count_params

model = GNO2d()                           # saved result: 176,001 parameters
print("parameters:", count_params(model))
out = model(torch.randn(2, 32, 32))
print("output:", tuple(out.shape))